# STEP1. 데이터 준비 (Data Preparation)

## 분석 질문
- 산업별 지수 · KOSPI 지수 · 국고채 3년/10년 금리 원본을 정제해 거래일 기준 하나의 패널로 병합하고, 결측치를 확인해 다음 STEP에서 쓸 수 있는 상태로 만든다.
- 관련 가설: 해당 없음 (H1·H2 공통 인프라)

## 입력
- 입력 파일: `data/raw/industry/*`, `data/raw/market/kospi_index/*`, `data/raw/rates/treasury_yield_3y_10y.csv`
- 이전 STEP 결과: 없음

## 출력
- Table: `data/interim/industry_daily_clean.csv`, `data/interim/kospi_daily.csv`, `data/interim/yields_daily.csv`
- Table: `data/processed/industry_rate_panel_raw.csv`
- Summary: 없음

## 실행 환경
- VS Code(로컬) 환경에서 실행한다.
- 원본 데이터는 `data/raw/`에 위치하며, 모든 처리 로직은 이 경로를 기준으로 동작한다.

## 다음 단계
- `02_variable_generation.ipynb`에서 이 노트북이 저장한 데이터를 불러와 산업별 수익률 · KOSPI 수익률 · 금리 변화분 · 장단기 금리차 등 파생변수를 생성한다.

## 참고
- 데이터 적재·정제·병합 함수는 `src/data_prep.py`에 정리되어 있으며, 이 노트북은 해당 함수를 호출해 각 단계를 확인하는 데 집중한다(계산 로직은 이전 버전 노트북과 동일).

## 준비: 실행 환경 설정
### 목적
- 저장소 루트를 자동으로 찾아 `config`, `src.data_prep`를 불러온다.

In [ ]:
# 저장소 루트를 찾아 config를 불러온다 (실행 위치/편집기와 무관하게 동작).
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더 (Jupyter/Lab/Colab)
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
# 마커를 하나만 보면 다른 폴더의 동명 config.py 를 잘못 집을 수 있다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
# 이미 sys.path에 있더라도 맨 앞으로 올린다. cwd(예: 바탕화면)가 앞에 있으면
# 같은 이름의 다른 config.py 가 먼저 잡히기 때문이다.
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )

from src import data_prep as dp

# 개인 PC 절대경로를 출력에 남기지 않는다 — 저장소 폴더명만 확인한다.
print("저장소 루트:", config.PROJECT_ROOT.name)

## 1. 데이터 불러오기
### 목적
- 산업별 지수(csv) + KOSPI 지수 + 국고채 3년·10년 금리 원본을 읽는다.
- 원본은 확장자(csv/xlsx)와 인코딩이 파일마다 달라 `read_table_safely`가 이를 자동으로 처리한다(기존 `read_csv_safely` 이름도 그대로 동작).

In [4]:
industry_daily, kospi_daily = dp.load_industry_and_kospi()
yields_daily = dp.load_yields()

print("industry_daily:", industry_daily.shape, "| 산업 수:", industry_daily["industry"].nunique())
print("kospi_daily   :", kospi_daily.shape)
print("yields_daily  :", yields_daily.shape)
industry_daily.head()

industry_daily: (17164, 4) | 산업 수: 7
kospi_daily   : (2452, 2)
yields_daily  : (2462, 3)


,date,industry,industry_index,source_file
0,2016-01-04,건설업,104.43,16-17.csv
1,2016-01-05,건설업,104.62,16-17.csv
2,2016-01-06,건설업,104.03,16-17.csv
3,2016-01-07,건설업,102.52,16-17.csv
4,2016-01-08,건설업,104.37,16-17.csv


## 2. 날짜 형식 통일
### 목적
- 세 데이터 모두 `date` 컬럼을 `datetime64`로 통일했는지 확인한다(변환 자체는 `load_industry_and_kospi`, `load_yields` 내부에서 수행됨).

In [5]:
print("industry_daily date dtype:", industry_daily["date"].dtype)
print("kospi_daily date dtype   :", kospi_daily["date"].dtype)
print("yields_daily date dtype  :", yields_daily["date"].dtype)

industry_daily date dtype: datetime64[us]
kospi_daily date dtype   : datetime64[us]
yields_daily date dtype  : datetime64[us]


## 3. 거래일 기준 병합
### 목적
- 산업지수 + KOSPI + 국고채 금리를 `date` 기준으로 병합한다(원본 레벨만 병합하며, 수익률·금리 변화분 등 파생변수는 `02_variable_generation.ipynb`에서 만든다).

In [6]:
raw_panel = dp.build_raw_panel(industry_daily, kospi_daily, yields_daily)
print("raw_panel:", raw_panel.shape)
raw_panel.head()

raw_panel: (17164, 7)


,date,industry,industry_index,kospi_index,y3,y10,source_file
0,2016-01-04,건설업,104.43,1918.76,1.634,2.039,16-17.csv
1,2016-01-04,금융업,408.22,1918.76,1.634,2.039,16-17.csv
2,2016-01-04,운수장비,1731.95,1918.76,1.634,2.039,16-17.csv
3,2016-01-04,의약품,8411.35,1918.76,1.634,2.039,16-17.csv
4,2016-01-04,전기전자,9239.12,1918.76,1.634,2.039,16.1~17.12.csv


## 4. 결측치 확인 및 처리
### 목적
- 병합된 원본 레벨 패널의 결측치를 확인한다. 산업/기간/컬럼 구성이 예상과 같은지도 함께 점검한다.

In [7]:
missing_check = dp.check_missing(raw_panel)
for k, v in missing_check.items():
    print(k, ":", v)

shape : (17164, 7)
columns : ['date', 'industry', 'industry_index', 'kospi_index', 'y3', 'y10', 'source_file']
industries : ['건설업', '금융업', '운수장비', '의약품', '전기전자', '철강금속', '화학']
date_range : (Timestamp('2016-01-04 00:00:00'), Timestamp('2025-12-30 00:00:00'))
n_missing : {'date': 0, 'industry': 0, 'industry_index': 0, 'kospi_index': 0, 'y3': 0, 'y10': 0, 'source_file': 0}


### 관찰 결과
- 원본 레벨(`industry_index`, `kospi_index`, `y3`, `y10`) 컬럼에는 결측치가 없다(값이 없는 행은 병합 전 단계에서 이미 제거됨).
- 산업 7개, 기간·행 수가 예상과 일치한다.
- 별도의 결측치 처리(대체/보간)는 필요하지 않다.

## 5. 최종 분석용 데이터 저장
### 목적
- 정제된 중간 데이터(산업별/KOSPI/국고채)와 병합된 원본 레벨 패널을 저장한다.

In [ ]:
industry_daily.to_csv(config.INTERIM_DIR / "industry_daily_clean.csv", index=False, encoding="utf-8-sig")
kospi_daily.to_csv(config.INTERIM_DIR / "kospi_daily.csv", index=False, encoding="utf-8-sig")
yields_daily.to_csv(config.INTERIM_DIR / "yields_daily.csv", index=False, encoding="utf-8-sig")

raw_panel_path = dp.save_raw_panel(raw_panel)

# 개인 PC 절대경로를 출력에 남기지 않는다 (config._rel: 저장소 루트 기준 상대경로).
print("저장 완료:")
print(" -", config._rel(config.INTERIM_DIR / "industry_daily_clean.csv"))
print(" -", config._rel(config.INTERIM_DIR / "kospi_daily.csv"))
print(" -", config._rel(config.INTERIM_DIR / "yields_daily.csv"))
print(" -", config._rel(raw_panel_path))

## STEP1 요약

### Takeaway
- 산업별/KOSPI/국고채 원본 데이터를 로드하고 날짜 형식을 통일했다.
- 거래일 기준으로 병합해 원본 레벨 패널(`industry_rate_panel_raw.csv`)을 만들었다.
- 결측치는 확인 결과 없음(사전 제거됨) — 별도 처리 불필요.
- 다음 단계: `02_variable_generation.ipynb`에서 `industry_rate_panel_raw.csv`(및 `kospi_daily.csv`, `yields_daily.csv`)를 불러와 파생변수를 생성한다.

In [ ]:
import pandas as pd

# 개인 PC 절대경로(sys.executable)는 출력에 남기지 않는다.
print("pandas:", pd.__version__)